# Single-chain model validation

In [ ]:
from functools import partial
import sys
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / "analysis" / "src" / "plots" / "style.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"

In [ ]:
from src.plots.figures import (
    plot_mfe_pairing_scatter, plot_precision_recall, plot_rg_validation,
)

SWEEP = "53c_single_mol_duplex"
DENSITY = None
RG_REFERENCE_NM = 1.47

replicates = load_sweep(OUTPUT_ROOT, SWEEP, "replicates")
summary = load_sweep(OUTPUT_ROOT, SWEEP, "summary")
if DENSITY is not None:
    replicates = filter_rows(replicates, density=DENSITY)
    summary = filter_rows(summary, density=DENSITY)
E_VALUES = sorted(summary["E_att"].dropna().unique())

## MFE-matching versus additional base pairs

In [ ]:
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(plot_mfe_pairing_scatter, style=STYLE, attraction_order=E_VALUES),
    style=STYLE, ncols=3, sharex=True, sharey=True, legend="none",
    height=3.5,
)
scale = max(replicates["mfe_matches"].max(), replicates["extra_pairs"].max()) * 1.08
for ax in axes.flat:
    ax.set_xlim(0, scale)
    ax.set_ylim(0, scale)
    ax.set_aspect("equal", adjustable="box")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, [rf"$E_{{\mathrm{{att}}}}={label}$" for label in labels],
           loc="lower center", bbox_to_anchor=(.5, -.06), ncol=len(labels),
           frameon=False, title=None)
plt.show()

## Precision and recall

In [ ]:
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(plot_precision_recall, style=STYLE, attraction_order=E_VALUES),
    style=STYLE, ncols=3, sharex=True, sharey=True,
    legend="none", height=3.5,
)
for ax in axes.flat:
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_aspect("equal", adjustable="box")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, title=r"$E_{\mathrm{att}}$",
           loc="lower center", bbox_to_anchor=(.5, -.06),
           ncol=len(labels), frameon=False)
plt.show()

## Radius of gyration

In [ ]:
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=lambda d, ax: plot_rg_validation(
        d, ax,
        replicates=filter_rows(replicates, angle_k_ppp=d["angle_k_ppp"].iloc[0]),
        style=STYLE, x_order=E_VALUES, reference_nm=RG_REFERENCE_NM,
    ),
    style=STYLE, ncols=3, sharex=True, sharey=True,
    legend="none", height=3.4,
)
rg_max = max(summary["Rg_nm_p90"].max(), RG_REFERENCE_NM) * 1.12
rg_min = max(0, min(summary["Rg_nm_p10"].min(), RG_REFERENCE_NM) - .12)
for ax in axes.flat:
    ax.set_ylim(rg_min, rg_max)
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([0], [0], color="#B85058", ls="--",
                           label=f"Reference ({RG_REFERENCE_NM:g} nm)")],
           loc="lower center", bbox_to_anchor=(.5, -.045), frameon=False)
plt.show()